<div style="background: linear-gradient(135deg, #fd7e14 0%, #dc3545 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">📉 02 — Introdução Teórica ao PCA</h1>
  <p style="color: #d1f2eb; margin: 8px 0 0 0; font-size: 1.1em;">Fase 01 · Bloco 05 · Autovalores</p>
</div>

## 🎯 Objetivos deste notebook

Ao finalizar este notebook você vai:
- Entender por que PCA depende de autovalores e autovetores
- Implementar PCA passo a passo com NumPy puro
- Visualizar as direções principais sobrepostas aos dados
- Reduzir dimensionalidade e medir variância explicada

---

## 1️⃣ O Problema: Dados com Dimensões Demais

Imagine um dataset com 100 features (colunas). Treinar com todas pode ser:
- **Lento** (muitos parâmetros)
- **Ruidoso** (features irrelevantes atrapalham)

PCA resolve isso encontrando as **direções de maior variância** e projetando os dados nessas direções, descartando o restante.

## 2️⃣ A Conexão: Covariância → Autovalores → PCA

O fluxo matemático é:
1. Centralizar os dados (subtrair a média)
2. Calcular a **matriz de covariância** `C`
3. Calcular **autovalores e autovetores** de `C`
4. Autovetores = direções principais
5. Autovalores = variância explicada em cada direção
6. Ordenar por autovalor decrescente → pegar os `k` maiores

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Gerar dados 2D correlacionados (para visualização)
np.random.seed(42)
media = [3, 5]
cov_real = [[2.0, 1.5],   # variância X=2, covariância=1.5
             [1.5, 1.0]]   # covariância=1.5, variância Y=1
dados = np.random.multivariate_normal(media, cov_real, size=200)

print(f'Shape dos dados: {dados.shape}')
print(f'Média por coluna: {dados.mean(axis=0).round(2)}')

## 3️⃣ PCA do Zero — Passo a Passo

In [ ]:
# Passo 1: Centralizar
dados_centralizados = dados - dados.mean(axis=0)

# Passo 2: Matriz de covariância
C = np.cov(dados_centralizados.T)  # shape (2, 2)
print('Matriz de Covariância:')
print(np.round(C, 4))

# Passo 3: Autovalores e autovetores
autovalores, autovetores = np.linalg.eig(C)
print(f'\nAutovalores: {autovalores.round(4)}')
print(f'Autovetores:\n{autovetores.round(4)}')

# Passo 4: Ordenar por autovalor decrescente
ordem = np.argsort(autovalores)[::-1]
autovalores = autovalores[ordem]
autovetores = autovetores[:, ordem]

# Variância explicada
variancia_explicada = autovalores / autovalores.sum() * 100
print(f'\nVariância explicada: PC1={variancia_explicada[0]:.1f}%, PC2={variancia_explicada[1]:.1f}%')

## 4️⃣ Visualizar os Componentes Principais

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Plotar dados
ax.scatter(dados_centralizados[:, 0], dados_centralizados[:, 1], 
           alpha=0.4, s=20, color='steelblue')

# Plotar autovetores (escalados pelo autovalor para visualização)
cores = ['red', 'orange']
for i in range(2):
    vetor = autovetores[:, i] * autovalores[i]  # escalar pelo autovalor
    ax.annotate('', xy=vetor, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=cores[i], lw=3))
    ax.text(vetor[0]*1.2, vetor[1]*1.2, 
            f'PC{i+1} ({variancia_explicada[i]:.1f}%)', 
            fontsize=12, color=cores[i], fontweight='bold')

ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('Dados + Componentes Principais', fontsize=14, fontweight='bold')
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
plt.show()

## 5️⃣ Projeção: Reduzir de 2D para 1D

Ao projetar nos `k` primeiros componentes, mantemos a **maior parte da informação**.

In [ ]:
# Projetar nos k=1 primeiros componentes
k = 1
W = autovetores[:, :k]  # Matriz de projeção (2, 1)
dados_projetados = dados_centralizados @ W  # (200, 1)

# Reconstruir de volta para 2D (para visualizar)
dados_reconstruidos = dados_projetados @ W.T

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(dados_centralizados[:, 0], dados_centralizados[:, 1], 
                alpha=0.4, s=20, color='steelblue')
axes[0].set_title('Original (2D)', fontsize=13, fontweight='bold')
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(dados_reconstruidos[:, 0], dados_reconstruidos[:, 1], 
                alpha=0.4, s=20, color='tomato')
axes[1].set_title(f'Reconstruído com {k} PC ({variancia_explicada[0]:.1f}% variância)', 
                  fontsize=13, fontweight='bold')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6️⃣ Validação: Comparar com sklearn

Agora vamos verificar se nosso PCA manual dá o mesmo resultado que a biblioteca:

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca.fit(dados_centralizados)

print('=== Comparação ===')
print(f'Nossos autovalores:    {autovalores.round(4)}')
print(f'Variância do sklearn:  {pca.explained_variance_.round(4)}')
print(f'\n% variância (nosso):  {variancia_explicada.round(2)}')
print(f'% variância (sklearn): {(pca.explained_variance_ratio_ * 100).round(2)}')
print(f'\nComponentes (nosso):\n{autovetores.round(4)}')
print(f'Componentes (sklearn):\n{pca.components_.T.round(4)}')
print('\n✅ Valores devem ser iguais (ou com sinal invertido — isso é normal em PCA).')

## 🏋️ Questões para Praticar

### Q1. PCA no Iris
Carregue o dataset `sklearn.datasets.load_iris()` (4 features). Aplique PCA do zero para reduzir a 2D. Plote colorido pelas 3 classes.

### Q2. Scree Plot
Calcule PCA no Iris com todas as 4 componentes. Faça um gráfico de barras da variância explicada por cada PC (Scree Plot). Quantas PCs capturam 95% da variância?

### Q3. Reconstrução com Perda
No Iris, reconstrua os dados usando apenas 1 PC, 2 PCs e 3 PCs. Calcule o **erro de reconstrução** (MSE entre original e reconstruído) para cada caso.

In [ ]:
# ============================================
# Resolva as questões aqui
# ============================================

# Q1: PCA no Iris


# Q2: Scree Plot


# Q3: Reconstrução com perda
